# 05.4 — Guided failure practice

[Class index and setup](README.md) · [Previous: 05.3](05.3-a-bounded-harness.ipynb) · [Next: 05.5](05.5-tests-evaluation-and-review.ipynb)

**Purpose:** Diagnose saved failures while keeping code changes tiny.

**In class:** 0 minutes. **Independent work:** 60 minutes, included in the index's 210-minute allowance. Preparation has its own 30 minutes. The instructor's project is additional.

**Starting knowledge:** Basic Python functions, dictionaries, JSON/schema validation, and the earlier pytest ideas. All code and saved inputs are supplied again; no earlier exercise answers or paid key are needed.

**Learning outcomes:** Locate the failing contract; distinguish missing data from failure; explain manual recovery.

[Glossary](GLOSSARY.md) · [Supplied code](report_app/__init__.py) · [Fixture provenance](fixtures/README.md)

In [ ]:
from pathlib import Path
import sys

if not Path("report_app/harness.py").is_file() or not Path("fixtures/world-bank-population.json").is_file():
    raise FileNotFoundError(
        "Open 05-small-ai-applications.code-workspace and restart the kernel. "
        "Download the complete class folder with report_app/, fixtures/, and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class assets available. Default execution is REPLAY, with no HTTP or credentials.")

## Independent work · 60 minutes

Use the first **15 minutes** for the guided walkthrough below, completing the total 60-minute walkthrough allocation. Exercise 05.4.H1 uses the next **45 minutes** for supplied tests and saved failure diagnosis. Nothing here requires a new tool, dispatcher, loop, model account, or live request.

### Guided walkthrough — 15 minutes

Spend five minutes following a two-tool comparison trace and checking the two numbers. Spend five reading the small `dispatch` function and the SDK boundary linked below. Spend five locating the application counters and the final `Answer` validation in `harness.py`. Read for responsibilities, not for memorizing every branch.

The provider module exposes a single request callable. The replay version returns a scripted envelope, checking actual tool results against saved expectations before returning the next response. The live version uses the installed OpenAI SDK with the same offered tools and answer schema. Both send their responses through the same supplied application loop.

In [ ]:
import inspect
from report_app.harness import run
from report_app.provider import make_replay, make_live_request
from report_app.source import read_fixture
from report_app.tools import dispatch

script = read_fixture("replays/compare.json")
comparison = run(script["prompt"], make_replay("compare"))
for event in comparison["trace"]:
    print(event)
print(inspect.getsource(dispatch))
print(inspect.getsource(make_live_request))  # Reading source sends nothing.

### Locate the contract that failed

| Outcome | What happened | First manual recovery step |
| --- | --- | --- |
| `answered` | Final JSON passed Answer validation | Check the claims against the actual tool results |
| `unknown_tool` | Name was absent from the allowlist | Inspect proposal/instructions; never automatically add the name |
| `invalid_arguments` | Arguments failed the permitted tool schema | Correct the label field/type; rerun the normal supplied case |
| `invalid_output` | Final JSON was missing, malformed, or wrong for Answer | Inspect the final text and expected field/type |
| `refused` | A provider refusal content item appeared | Inspect the request and explain the refusal |
| `incomplete` | Provider response did not finish | Inspect the output budget; retry manually if appropriate |
| `api_error` | SDK reported a transport or HTTP error | Check configuration/service/access; recover with replay |
| `tool_limit` | A third execution was proposed | Inspect why the model repeated work; no additional execution occurred |
| `model_limit` | Request budget ended without a final answer | Inspect the trace; no request beyond the budget was sent |
| `invalid_response` | Unexpected status/shape or simultaneous calls | Inspect response against the one-call contract |
| `replay_error` | Script ran out or actual tool result differed | Check the selected script and Python operation; do not label it live evidence |

A valid exact lookup for an absent country is **not** one of these failure outcomes. It returns an empty list. The application can then answer that the saved data is absent. Missing is not zero, and a connection error must not be converted to an empty result.

The harness stops the current run on failure. There is no automatic repair loop, retry, fallback tool, or silent increase in limits. Its live SDK client also disables hidden retries. A timeout is a socket setting, not a promise of a strict total run duration. All of this behavior is supplied code.

In [ ]:
for case in ["unknown-tool", "invalid-arguments", "malformed", "refused", "incomplete", "repeated"]:
    script = read_fixture(f"replays/{case}.json")
    result = run(script["prompt"], make_replay(case))
    print(case, "→", result["outcome"], "requests:", result["model_requests"], "tools:", result["tool_executions"])

### Visual guide — Locate the first failed boundary

<img src="images/failure-layers.png" alt="Unknown names stop at permission checks; invalid arguments stop at argument validation; valid tools may return no matches; malformed final output stops at Answer validation." width="395" style="max-width: 100%; height: auto; background-color: white;">

An unknown name and invalid arguments stop before a tool runs. A valid missing-country lookup can return an empty list. A malformed final answer fails a later boundary. Diagnose the responsible layer rather than treating every case as a prompt problem.

*Credit: Original ADPRO course diagram (2026), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). [Editable Mermaid source](images/failure-layers.mmd); [sources and teaching notes](IMAGE_SUGGESTIONS.md).*

### Three descriptions that should not be confused

1. The `modify` replay contains an ordinary schema-valid answer saying that writes are unavailable. It uses zero tools and finishes as `answered`.
2. The `refused` replay contains a provider `refusal` item. It finishes as `refused` without accepting a final answer.
3. The `unknown-tool` replay proposes a forbidden function. The dispatcher blocks it regardless of what the text says.

Similarly, invalid argument JSON and invalid final answer JSON are validated at different boundaries. The correct repair belongs to the failing contract. Asking a model to “be better” does not fix a missing application limit or add a runtime validation check.

```python
script = read_fixture("replays/invalid-arguments.json")
result = run(script["prompt"], make_replay("invalid-arguments"))
expected_outcome = ____  # Fill one quoted string from the table.
assert result["outcome"] == expected_outcome
assert result["tool_executions"] == 0
```

Focused supplied checks, run manually in the class terminal:

```sh
uv run --locked python -m pytest -q -k 'saved_failures or argument_contract or no_argument'
```

For the repair comparison, rerun the complete supplied `portugal` script with its valid string label. A one-step failure script does not contain the later responses needed for a successful run.

### Exercise 05.4.H1 — Name the failure

> **Independent work · 45 minutes, included in this part’s 60**
>
> **Goal:** Use supplied traces/tests to locate failures with a one-string completion.
>
> **Start here:** The outcome table and cases above; copy the snippet above.
>
> **Task:** Spend 10 minutes predicting unknown-tool/invalid-arguments/malformed outcomes, 10 checking the supplied traces, 10 completing the single outcome string and running the supplied focused tests, 10 comparing an absent-country result with invalid input, and 5 explaining a repair. Do not implement a repair loop.
>
> **Check:** The invalid-argument run has the expected outcome and zero tool executions; distinguish an empty lookup from an invalid call.
>
> **Hint:** The outcome string is `invalid_arguments`. Correcting label 123 means supplying a nonempty string, not permitting numeric arguments.

In [ ]:
# Your attempt for 05.4.H1.
# Complete only the tiny expression; record observations/explanations here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_05_4_h1.py

## Completion and reflection — included above

Record one rejected proposal, the contract that rejected it, and a manual next step. Explain why a failed request cannot be represented as a real population of zero. Follow [AI_SETUP.md](AI_SETUP.md) only if explicitly exploring live access; required completion remains replay-based.

[Failure tests](tests/test_harness.py) · [Supplied SDK/replay boundary](report_app/provider.py) · [Glossary](GLOSSARY.md)